# RAG Evaluation with RAGAS
## Faithfulness and Relevancy — Measuring RAG Quality
⏱ ~12 min

Building a RAG pipeline is only half the job. How do you know it is actually working well?
**RAGAS** (Retrieval Augmented Generation Assessment) gives you automatic metrics that score your pipeline without needing manually labelled data for every question.

By the end of this workshop you will:
- Understand what RAGAS measures and why each metric matters
- Build a RAG pipeline and run a full evaluation against three metrics
- Deliberately break the pipeline and watch the scores drop
- Know exactly which knob to turn when each score is low

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why evaluate? What do RAGAS metrics measure? |
| 2 | **Build the Pipeline** — RAG system under test |
| 3 | **Evaluation Dataset** — Capturing the full trace |
| 4 | **Running RAGAS** — Scores, DataFrames, and reading results |
| 5 | **Diagnosing Failures** — What low scores tell you |
| 6 | **Degradation Experiments** — Prove the metrics work |
| ★ | **Exercises + Answer Key** (bonus) |

---

### Prerequisites
- Python 3.10+ (a `.venv` with requirements already installed, or Colab)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- Recommended: complete **Example 12 — Basic RAG Notebook** first

### Key References
> Es, S., James, J., et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation.* https://arxiv.org/abs/2309.15217  
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401  
> Chang, Y., et al. (2024). *A Survey on Evaluation of Large Language Models.* ACM TIST. https://arxiv.org/abs/2307.03109

In [ ]:
# Install dependencies -- runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "ragas",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "python-dotenv",
            "datasets",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected -- skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon), name = OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: add it to .env  |  Colab: add it to Secrets panel"
)
print(f"OPENAI_API_KEY loaded: {key[:8]}...{key[-4:]}")

---

## Part 1 — Concepts: Why Evaluate a RAG Pipeline?


---

### The Problem with Eyeballing

When you build a RAG pipeline, it is tempting to run a few queries, see reasonable-looking answers, and call it done. This fails in production because:

1. **Invisible hallucinations** — The model blends retrieved context with its own training data. The answer looks correct but contains fabricated details.
2. **Retrieval drift** — A change in chunk size, embedding model, or k causes recall to drop silently.
3. **No regression detection** — You cannot tell if a new prompt template made things better or worse without numbers.

Evaluation gives you **reproducible, automatic scores** so every change is measurable.

---

### The RAG Evaluation Pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│                        RAG SYSTEM UNDER TEST                        │
│                                                                     │
│   Documents  →  Vector Store  →  Retriever  →  LLM  →  Answer      │
└──────────────────────────────┬──────────────────────────────────────┘
                               │  capture full trace
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                         EVALUATION DATASET                          │
│                                                                     │
│   user_input  │  reference  │  retrieved_contexts  │  response      │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                          RAGAS METRICS                              │
│                                                                     │
│   Faithfulness  │  Answer Relevance  │  Context Recall  │  Precision│
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
               Scores 0.0 – 1.0  (higher = better)
                               │
                               ▼
             Diagnose  →  Fix chunk size / k / prompt / model
```

---

### The Four Core RAGAS Metrics

| Metric | What it measures | Input required | Formula |
|--------|-----------------|---------------|--------|
| **Faithfulness** | Every claim in the answer is grounded in the retrieved context | question, contexts, response | claims supported by context / total claims |
| **Answer Relevance** | The answer actually addresses the question | question, response | cosine_sim(generated questions from answer, original question) |
| **Context Recall** | Retrieved context covers the ground truth answer | question, reference, contexts | ground truth sentences supported by context / total |
| **Context Precision** | Retrieved context is focused (not noisy) | question, reference, contexts | relevant chunks at top / total retrieved chunks |

All scores are in **[0.0, 1.0]**. Higher is always better. A score of 1.0 is perfect.

---

### Failure Mode → Metric Mapping

| Failure mode | Symptom you see | Metric that catches it | Fix |
|---|---|---|---|
| LLM makes up facts not in the context | Confident but wrong answers | **Faithfulness** ↓ | Tighten system prompt, lower temperature |
| Answer doesn't address the question | Technically true but irrelevant | **Answer Relevance** ↓ | Enforce concise, direct answers in prompt |
| Retriever misses necessary documents | Incomplete answers | **Context Recall** ↓ | Increase k, reduce chunk size |
| Retriever pulls noisy unrelated chunks | Distracted LLM, vague answers | **Context Precision** ↓ | Reduce k, filter by metadata, use MMR |

---

### RAGAS vs Other Evaluation Frameworks

| Framework | Approach | Requires ground truth | Reference-free | Best for |
|-----------|----------|----------------------|----|----------|
| **RAGAS** | LLM-as-judge with specialized metrics | Partial (recall only) | Faithfulness, Relevance | RAG-specific evaluation |
| **DeepEval** | LLM-as-judge, many metrics | Optional | Yes | General LLM evaluation |
| **ROUGE** | N-gram overlap with reference | Always | No | Summarization benchmarks |
| **BERTScore** | Embedding similarity to reference | Always | No | Translation, summarization |
| **TruLens** | RAG triad (same as RAGAS core) | Optional | Yes | LangChain/LlamaIndex apps |

**Why RAGAS for this workshop?** It is purpose-built for RAG, works with LangChain out of the box, and its Faithfulness + Answer Relevance metrics need *no* ground truth — just the question and the answer.

---

## Part 2 — Build the RAG Pipeline Under Test


---

Before evaluating anything, we need a working RAG pipeline. We use a small hardcoded knowledge base about Python so that correct answers are easy to verify by hand — the same principle applies to any domain.

The pipeline follows the exact LCEL pattern from Example 12:

```
Documents → Embed → VectorStore       (offline, run once)
Question  → Retriever → Prompt → LLM → Answer    (online)
```

In [ ]:
# ─── 2-A: Define the knowledge base ──────────────────────────────────────────
# Small, hand-written documents so we can verify retrieval by inspection.
# In production this would be a real corpus of PDFs, docs, etc.

from langchain_core.documents import Document

DOCS = [
    Document(
        page_content=(
            "Python lists are ordered, mutable sequences. "
            "They support indexing, slicing, and methods like "
            "append(), extend(), and pop()."
        ),
        metadata={"source": "python-basics", "topic": "lists"},
    ),
    Document(
        page_content=(
            "Python dictionaries store key-value pairs. "
            "Keys must be hashable. Common methods include "
            "get(), keys(), values(), items(), and update()."
        ),
        metadata={"source": "python-basics", "topic": "dicts"},
    ),
    Document(
        page_content=(
            "Python functions are defined with the def keyword. "
            "They support default arguments, *args for variable positional "
            "arguments, and **kwargs for keyword arguments."
        ),
        metadata={"source": "python-functions", "topic": "functions"},
    ),
    Document(
        page_content=(
            "List comprehensions provide a concise way to create lists: "
            "[expr for item in iterable if condition]. "
            "They are generally faster than equivalent for loops."
        ),
        metadata={"source": "python-advanced", "topic": "comprehensions"},
    ),
    Document(
        page_content=(
            "Python classes are defined with the class keyword. "
            "The __init__ method initializes instances. "
            "Python supports single and multiple inheritance."
        ),
        metadata={"source": "python-oop", "topic": "classes"},
    ),
    Document(
        page_content=(
            "Python exceptions are handled with try/except/finally blocks. "
            "Common exceptions include ValueError, TypeError, "
            "KeyError, and IndexError."
        ),
        metadata={"source": "python-errors", "topic": "exceptions"},
    ),
]

print(f"Knowledge base: {len(DOCS)} documents")
for doc in DOCS:
    print(f"  [{doc.metadata['topic']:16}] {doc.page_content[:55]}...")

In [ ]:
# ─── 2-B: Embed and build the vector store ───────────────────────────────────
# Using in-memory Chroma for simplicity -- no disk I/O needed for evaluation.

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(DOCS, embedding=embeddings)
# k=2: retrieve 2 chunks per query — tradeoff: higher k improves recall but adds noise and raises token cost
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Quick sanity check -- confirm retrieval returns sensible chunks
test_results = retriever.invoke("How do Python lists work?")
print(f"Retriever smoke test: {len(test_results)} chunks returned")
for r in test_results:
    print(f"  [{r.metadata['topic']}] {r.page_content[:70]}...")

In [ ]:
# ─── 2-C: Build the LCEL RAG chain ───────────────────────────────────────────
# Standard pattern: retriever | format_docs feeds context,
# RunnablePassthrough() feeds the question unchanged.

# StrOutputParser strips the AIMessage wrapper so rag_chain returns a plain string, not a message object
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")

# The grounding constraint in the system prompt is the key variable
# we will manipulate in Part 6 to show faithfulness dropping.
SYSTEM_PROMPT = (
    "Answer the question using ONLY the context below. "
    "If the context does not contain the answer, say 'I don't know'.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


rag_chain = (
# LCEL pipe (|) wires stages lazily — nothing runs until .invoke(); RunnablePassthrough forwards the raw question unchanged
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Smoke test
answer = rag_chain.invoke("What methods do Python lists support?")
print("Smoke test answer:")
print(answer)

---

## Part 3 — Building the Evaluation Dataset


---

RAGAS needs a **dataset** — a table where each row is one question with all the evidence attached. Here is exactly what goes in each column:

| Column | What it is | Who provides it |
|--------|------------|----------------|
| `user_input` | The question asked | You (test set) |
| `reference` | The ideal ground-truth answer | You (written by hand) |
| `retrieved_contexts` | List of chunks the retriever returned | The pipeline (captured) |
| `response` | What the RAG chain actually answered | The pipeline (captured) |

> **Why write ground truth by hand?** Faithfulness and Answer Relevance do not need it — RAGAS generates its own internal reference. But **Context Recall** needs `reference` to measure whether the retriever covers all necessary facts. Without it, Context Recall cannot run.

**Key principle:** Ground truth should reflect what the knowledge base *actually says*, not what the ideal answer should be in general. If the knowledge base only says "lists support append(), extend(), pop()", the reference should say the same — not "all 20+ list methods".

In [ ]:
# ─── 3-A: Test questions and ground truth ─────────────────────────────────────
# Ground truth answers are written to match what the knowledge base says.
# They should NOT exceed what is in DOCS -- RAGAS measures retrieval coverage,
# not general Python knowledge.

TEST_SET = [
    {
        "question": "What methods do Python lists support?",
        "ground_truth": (
            "Python lists support methods like append(), extend(), and pop(), "
            "as well as indexing and slicing."
        ),
    },
    {
        "question": "How are Python dictionaries structured?",
        "ground_truth": (
            "Python dictionaries store key-value pairs where keys must be hashable. "
            "Common methods include get(), keys(), values(), items(), and update()."
        ),
    },
    {
        "question": "How do you define a Python function with variable arguments?",
        "ground_truth": (
            "Python functions are defined with def and support *args for variable "
            "positional arguments and **kwargs for keyword arguments."
        ),
    },
    {
        "question": "What is a list comprehension?",
        "ground_truth": (
            "A list comprehension creates a list using the syntax "
            "[expr for item in iterable if condition] "
            "and is generally faster than an equivalent for loop."
        ),
    },
    {
        "question": "How does Python handle exceptions?",
        "ground_truth": (
            "Python handles exceptions with try/except/finally blocks. "
            "Common exceptions include ValueError, TypeError, KeyError, and IndexError."
        ),
    },
]

print(f"Evaluation dataset: {len(TEST_SET)} questions")
for i, item in enumerate(TEST_SET, 1):
    print(f"  [{i}] {item['question']}")

In [ ]:
# ─── 3-B: Run the pipeline and capture the full trace ─────────────────────────
# For each question we capture:
#   - the retrieved chunks (before the LLM sees them)
#   - the final answer (after the LLM processes them)
# This is the "trace" RAGAS needs.

eval_rows = []

for item in TEST_SET:
    question = item["question"]

    # Step 1: retrieve chunks independently so we can inspect them
    contexts = retriever.invoke(question)

    # Step 2: run the full RAG chain to get the answer
    response = rag_chain.invoke(question)

    eval_rows.append(
        {
            "user_input": question,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in contexts],
            "response": response,
        }
    )

    print(f"Q: {question[:60]}")
    print(f"A: {response[:80]}...")
    print(f"   Retrieved {len(contexts)} chunks: {[c.metadata['topic'] for c in contexts]}")
    print()

print(f"Captured {len(eval_rows)} rows for evaluation.")

---

## Part 4 — Running RAGAS


---

### How RAGAS Scores Faithfulness (Under the Hood)

RAGAS uses an LLM internally to measure faithfulness. Here is the algorithm:

```
1. Extract all factual claims from the response
   e.g., ["lists support append()", "lists are mutable", "lists support slicing"]

2. For each claim, ask: can this be inferred from the retrieved context?
   e.g., [True, True, True]  → 3 / 3 supported

3. Faithfulness = supported_claims / total_claims  → 1.0
```

If the model adds "lists also support sorting with sorted()" but the retrieved chunk does not mention it, that claim is unsupported → Faithfulness < 1.0.

### How RAGAS Scores Answer Relevance

```
1. From the response, generate N synthetic questions that the response answers
   (e.g., "What methods do Python lists support?" from our answer)

2. Embed each synthetic question and the original question

3. Answer Relevance = mean cosine_similarity(synthetic_q, original_q)
```

An off-topic or vague answer generates synthetic questions that diverge from the original → low relevance score.

### How RAGAS Scores Context Recall

```
1. Break the reference (ground truth) into sentences
   e.g., ["lists support append()", "lists support indexing"]

2. For each sentence, check: is this covered by the retrieved context?

3. Context Recall = covered_sentences / total_reference_sentences
```

If k=1 and the single retrieved chunk misses a fact in the reference, Context Recall drops.

In [ ]:
# ─── 4-A: Prepare the RAGAS dataset ──────────────────────────────────────────
# RAGAS uses HuggingFace Datasets format as input.

from datasets import Dataset
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import answer_relevancy, context_recall, faithfulness

# Wrap LangChain objects so RAGAS can call them
# LangchainLLMWrapper/EmbeddingsWrapper bridge RAGAS internals to LangChain objects — RAGAS calls them via its own protocol
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# Convert our list of dicts to a HuggingFace Dataset
dataset = Dataset.from_list(eval_rows)

print("Dataset schema:")
print(dataset)
print("\nColumn names:", dataset.column_names)
print("\nFirst row preview:")
row = dataset[0]
print(f"  user_input:          {row['user_input']}")
print(f"  retrieved_contexts:  {len(row['retrieved_contexts'])} chunks")
print(f"  response (first 80): {row['response'][:80]}")

In [ ]:
# ─── 4-B: Run RAGAS evaluation ────────────────────────────────────────────────
# This makes LLM calls internally for each metric on each row.
# Expect roughly 15-25 API calls total for 5 questions x 3 metrics.

# faithfulness: does the answer contain only claims verifiable in the retrieved chunks? (LLM-graded)
# answer_relevancy: is the answer on-topic for the question? (embedding similarity-based)
# context_recall: does the retriever surface the chunks needed to answer? (compared against ground truth)
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n" + "=" * 55)
print("RAGAS EVALUATION RESULTS")
print("=" * 55)
print(results)

In [ ]:
# ─── 4-C: Per-question breakdown ──────────────────────────────────────────────
# Convert to pandas for readable per-row analysis.

df = results.to_pandas()

METRIC_COLS = ["faithfulness", "answer_relevancy", "context_recall"]

print("Per-question scores:")
print()
display_cols = ["user_input"] + METRIC_COLS
print(df[display_cols].to_string(index=False, max_colwidth=45))

print()
print("Mean scores:")
for col in METRIC_COLS:
    bar = "\u2588" * int(df[col].mean() * 20)
    print(f"  {col:<22} {df[col].mean():.3f}  {bar}")

---

## Part 5 — Reading and Diagnosing Scores


---

### Score Interpretation Table

| Score range | Interpretation | Action |
|-------------|---------------|--------|
| **0.9 – 1.0** | Production-ready for this question type | Ship it; monitor for regressions |
| **0.7 – 0.9** | Acceptable; some edge cases may fail | Investigate the low-scoring rows |
| **0.5 – 0.7** | Needs improvement | Diagnose using per-row breakdown |
| **< 0.5** | Serious problem | Check retrieval and prompt first |

### Metric Direction — What Hurts Each Score

| Metric | Hurts the score | Helps the score |
|--------|----------------|----------------|
| Faithfulness | LLM adds facts beyond the retrieved context | Stricter grounding prompt, lower temp |
| Answer Relevance | Answer is vague, off-topic, or repetitive | Concise prompt; direct answer instruction |
| Context Recall | Retriever misses key document | Higher k, smaller chunk_size, hybrid search |
| Context Precision | Retriever returns noisy unrelated chunks | Lower k, metadata filter, MMR retrieval |

> **Rule of thumb:** Fix retrieval problems first (recall, precision), then fix generation problems (faithfulness, relevance). A great LLM cannot compensate for missing context.

In [ ]:
# ─── 5-A: Find the weakest question per metric ───────────────────────────────
# Pinpoint exactly which question causes each metric to drop.

print("Worst-performing question per metric:\n")
for metric in METRIC_COLS:
    worst = df.nsmallest(1, metric).iloc[0]
    resp_col = "response" if "response" in df.columns else "answer"
    print(f"  Lowest {metric}: {worst[metric]:.3f}")
    print(f"  Question: {worst['user_input']}")
    print(f"  Answer:   {str(worst.get(resp_col, ''))[:120]}")
    print()

In [ ]:
# ─── 5-B: Inspect what the retriever actually returned ────────────────────────
# The most common debugging mistake is skipping this step.
# Low context_recall almost always means the retriever missed something obvious.

print("=== RETRIEVED CONTEXTS FOR EACH QUESTION ===")
print()

for row in eval_rows:
    print(f"Q: {row['user_input']}")
    for i, ctx in enumerate(row["retrieved_contexts"], 1):
        print(f"  [{i}] {ctx[:100]}...")
    print(f"  Reference: {row['reference'][:80]}...")
    print()

---

## Part 6 — Degradation Experiments


---

The most convincing way to trust a metric is to **deliberately break something** and confirm the score drops in the predicted direction. We run two targeted experiments:

1. **k=1 experiment** — halve retrieval coverage → Context Recall should drop  
2. **Loose prompt experiment** — remove grounding constraint → Faithfulness should drop

If RAGAS is working correctly, each experiment produces exactly the expected degradation.

In [ ]:
# ─── 6-A: k=1 experiment — cut retrieval coverage in half ────────────────────
# With k=1 the retriever only returns 1 chunk instead of 2.
# Questions that need evidence from 2 chunks will have incomplete context.
# Prediction: context_recall drops; faithfulness and answer_relevancy stay similar.

# Changing only k isolates retrieval as the variable — prompt and LLM are unchanged so score shifts trace back to retrieval
retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})

rag_chain_k1 = (
    {"context": retriever_k1 | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

eval_rows_k1 = []
for item in TEST_SET:
    q = item["question"]
    ctxs = retriever_k1.invoke(q)
    resp = rag_chain_k1.invoke(q)
    eval_rows_k1.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

results_k1 = evaluate(
    Dataset.from_list(eval_rows_k1),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
df_k1 = results_k1.to_pandas()

print("k=2 (original) vs k=1 (degraded):\n")
for metric in METRIC_COLS:
    orig = df[metric].mean()
    degrad = df_k1[metric].mean()
    delta = degrad - orig
    arrow = "\u2193" if delta < -0.02 else ("\u2191" if delta > 0.02 else "\u2192")
    print(f"  {metric:<22} k=2: {orig:.3f}  k=1: {degrad:.3f}  {arrow} {delta:+.3f}")

In [ ]:
# ─── 6-B: Loose prompt experiment — remove the grounding constraint ───────────
# Removing "ONLY" lets the LLM blend its own knowledge with retrieved context.
# The LLM adds extra details from its training data that are not in the chunks.
# Prediction: faithfulness drops; context_recall stays similar.

LOOSE_PROMPT = (
    "Answer the question. Use the context below if helpful.\n\n"
    "Context:\n{context}"
)

prompt_loose = ChatPromptTemplate.from_messages(
    [
        ("system", LOOSE_PROMPT),
        ("human", "{question}"),
    ]
)

# Same retriever and LLM — only the prompt changes; this isolates prompt wording as the single variable
rag_chain_loose = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_loose
    | llm
    | StrOutputParser()
)

eval_rows_loose = []
for item in TEST_SET:
    q = item["question"]
    ctxs = retriever.invoke(q)
    resp = rag_chain_loose.invoke(q)
    eval_rows_loose.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

results_loose = evaluate(
    Dataset.from_list(eval_rows_loose),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
df_loose = results_loose.to_pandas()

print("Strict prompt vs Loose prompt:\n")
for metric in METRIC_COLS:
    orig = df[metric].mean()
    degrad = df_loose[metric].mean()
    delta = degrad - orig
    arrow = "\u2193" if delta < -0.02 else ("\u2191" if delta > 0.02 else "\u2192")
    print(f"  {metric:<22} strict: {orig:.3f}  loose: {degrad:.3f}  {arrow} {delta:+.3f}")

print()
print("Expected: faithfulness drops with loose prompt (LLM adds facts from training data)")

In [ ]:
# ─── 6-C: Full comparison summary ────────────────────────────────────────────
# Side-by-side view of all three configurations.

print("=" * 65)
print(f"{'Configuration':<25} {'Faithfulness':>13} {'Ans. Relevance':>15} {'Ctx. Recall':>12}")
print("-" * 65)

configs = [
    ("k=2, strict prompt",  df),
    ("k=1, strict prompt",  df_k1),
    ("k=2, loose prompt",   df_loose),
]

for name, frame in configs:
    f  = frame["faithfulness"].mean()
    ar = frame["answer_relevancy"].mean()
    cr = frame["context_recall"].mean()
    print(f"{name:<25} {f:>13.3f} {ar:>15.3f} {cr:>12.3f}")

print("=" * 65)
print()
print("Observations:")
print("  k=1 primarily hurts Context Recall (fewer chunks = less coverage)")
print("  Loose prompt primarily hurts Faithfulness (LLM adds unchecked claims)")
print("  Answer Relevance is relatively stable -- the questions are on-topic either way")

---

## Part 7 \u2605 — Exercises


---

Try these before looking at the answer key. Each exercise is designed to build one specific intuition about how RAGAS responds to pipeline changes. The answer key is at the end of the notebook.

In [ ]:
# ── Exercise 1 — Change k and observe all three metrics ──────────────────────
#
# Run the evaluation with k=3 (more chunks than the original k=2).
# Predict before you run: which metric should improve? Which might get worse?
#
# After you run it, compare to the k=2 baseline:
#   - Did context_recall go up? (more coverage)
#   - Did faithfulness stay stable? (same prompt)
#   - Was there any change? (more noise is possible)

# TODO: create retriever_k3 with search_kwargs={"k": 3}
retriever_k3 = None  # replace with your retriever

# TODO: rebuild the chain using retriever_k3
rag_chain_k3 = None  # replace with your chain

# TODO: collect eval_rows_k3 the same way as the baseline
# TODO: call evaluate() and print the mean scores
# TODO: compare to df[METRIC_COLS].mean()

In [ ]:
# ── Exercise 2 — Add a new document and test it ───────────────────────────────
#
# Add a new Document to DOCS about Python generators.
# Add a matching question + ground_truth to the test set.
# Rebuild the vectorstore, re-run the pipeline, and evaluate ONLY the new row.
#
# Prediction: all three scores should be high if the new document is indexed correctly.
# If context_recall is low, the retriever is not surfacing your new chunk.

# TODO: define new_doc
new_doc = None  # Document(page_content="...", metadata={"source": "python-advanced", "topic": "generators"})

# TODO: rebuild vectorstore_ext = Chroma.from_documents(DOCS + [new_doc], embedding=embeddings)
# TODO: add question + ground_truth for generators to TEST_SET
# TODO: run the pipeline on the new question only and evaluate the single row

In [ ]:
# ── Exercise 3 — Build your own RAG pipeline and evaluate it ─────────────────
#
# Create a knowledge base on any topic you like (cooking, history, your docs).
# Write at least 5 documents, 4 test questions, and their ground truth answers.
# Run a full RAGAS evaluation on your pipeline.
#
# Goal: achieve faithfulness >= 0.8 and context_recall >= 0.7.
# If you can't hit those numbers, use the score breakdown to diagnose why.

my_docs = [
    # TODO: add at least 5 documents on your topic
]

my_test_set = [
    # TODO: add at least 4 questions + ground_truth answers
]

# TODO: build vectorstore, retriever, rag_chain
# TODO: collect eval_rows_mine
# TODO: call evaluate() and print results

---

## What's Next?

You now have a complete RAG evaluation workflow. Here is where to go deeper:

### Improve what you measured
- **Increase k** when Context Recall is low — more retrieved chunks means more coverage
- **Reduce chunk size** when retrieval returns too much noise — smaller chunks match more precisely
- **Add MMR retrieval** when Context Precision is low — `vectorstore.as_retriever(search_type="mmr")` diversifies the retrieved chunks
- **Tighten the system prompt** when Faithfulness is low — add "Do NOT add information that is not in the context"

### Extend the evaluation
- **Context Precision** — `from ragas.metrics import context_precision` — requires `reference`; measures whether retrieved chunks are focused
- **Answer Correctness** — `from ragas.metrics import answer_correctness` — requires `reference`; full accuracy check
- **CI integration** — run RAGAS as part of your test suite; fail the build if any score drops below a threshold
- **Synthetic test set generation** — `from ragas.testset import TestsetGenerator` — RAGAS can generate `(question, ground_truth)` pairs automatically from your documents

### Related examples in this repo
- **Example 12** — [Basic RAG Notebook](../12-basic-rag-notebook/rag_workbook.ipynb) — the pipeline architecture this notebook evaluates
- **Example 17** — Corrective RAG — automatically fix low-confidence retrievals at query time
- **Example 25** — Adaptive RAG — route queries to different retrieval strategies based on query type

### Further reading
- RAGAS documentation: https://docs.ragas.io
- LangChain RAGAS integration: https://docs.ragas.io/en/stable/howtos/integrations/langchain.html
- LangSmith tracing: https://docs.smith.langchain.com

---

*Workshop complete. You have a repeatable evaluation harness — run it every time you change chunking, embedding model, prompt, or k.*

---

## Answer Key

Review these only after attempting the exercises yourself. The goal of the exercises is not to produce a specific number — it is to build intuition about what moves each metric.

---

### Exercise 1 — k=3 vs k=2

**What to expect:** Increasing k from 2 to 3 gives the retriever one more chunk to work with. For questions whose ground truth spans multiple chunks, Context Recall should improve or stay the same — it never hurts to have more context, up to a point. Faithfulness typically stays stable because the grounding prompt has not changed. Answer Relevance is usually unaffected because the extra chunk rarely makes the answer less focused.

**If you see an unexpected drop in faithfulness:** the third chunk may contain related but slightly different content that confuses the LLM. This is a sign that Context Precision is a concern and MMR retrieval is worth trying.

**Key insight:** More chunks is not always better. The sweet spot is enough coverage for good recall without so much noise that faithfulness suffers.

In [ ]:
# Exercise 1 — sample solution: k=3 evaluation
retriever_k3 = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain_k3 = (
    {"context": retriever_k3 | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

eval_rows_k3 = []
for item in TEST_SET:
    q = item["question"]
    ctxs = retriever_k3.invoke(q)
    resp = rag_chain_k3.invoke(q)
    eval_rows_k3.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

results_k3 = evaluate(
    Dataset.from_list(eval_rows_k3),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
df_k3 = results_k3.to_pandas()

print("k=1 / k=2 / k=3 comparison:\n")
print(f"{'Metric':<22} {'k=1':>8} {'k=2':>8} {'k=3':>8}")
print("-" * 50)
for metric in METRIC_COLS:
    v1 = df_k1[metric].mean()
    v2 = df[metric].mean()
    v3 = df_k3[metric].mean()
    print(f"{metric:<22} {v1:>8.3f} {v2:>8.3f} {v3:>8.3f}")

### Exercise 2 — Add a New Document

**What to expect:** If the generator document is indexed correctly and the question is well-targeted, all three scores should be high for the new row. A low Context Recall on the new question means the retriever is not surfacing the new chunk — check that the question wording is close to the document content, and try increasing k.

**Common mistake:** Writing a ground truth answer that is more detailed than the indexed document. RAGAS will mark the extra sentences as not covered by the retrieved context, lowering Context Recall even when retrieval is working correctly.

In [ ]:
# Exercise 2 — sample solution: add a generators document
new_doc = Document(
    page_content=(
        "Python generators use the yield keyword to produce values lazily. "
        "They are memory-efficient for large sequences and can be iterated "
        "with for loops or the next() function."
    ),
    metadata={"source": "python-advanced", "topic": "generators"},
)

vectorstore_ext = Chroma.from_documents(DOCS + [new_doc], embedding=embeddings)
retriever_ext = vectorstore_ext.as_retriever(search_kwargs={"k": 2})

rag_chain_ext = (
    {"context": retriever_ext | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

new_question = {
    "question": "What is a Python generator and how is it used?",
    "ground_truth": (
        "Python generators use the yield keyword to produce values lazily. "
        "They are memory-efficient and can be iterated with for loops or next()."
    ),
}

q = new_question["question"]
ctxs = retriever_ext.invoke(q)
resp = rag_chain_ext.invoke(q)

new_row = [
    {
        "user_input": q,
        "reference": new_question["ground_truth"],
        "retrieved_contexts": [c.page_content for c in ctxs],
        "response": resp,
    }
]

print(f"Retrieved chunks for '{q}':")
for c in ctxs:
    print(f"  [{c.metadata['topic']}] {c.page_content[:80]}...")

result_new = evaluate(
    Dataset.from_list(new_row),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print("\nScores for the new generators question:")
print(result_new)

### Exercise 3 — Build Your Own Pipeline

**What good results look like:**
- Faithfulness >= 0.8 — the LLM stays grounded in what you indexed
- Answer Relevance >= 0.85 — answers directly address each question
- Context Recall >= 0.7 — retriever covers most of the ground truth

**If you cannot hit these targets:**
1. **Low Faithfulness** — Tighten the system prompt. Add "ONLY" and "Do NOT add information not in the context."
2. **Low Answer Relevance** — Your questions may be ambiguous, or the answers may be drifting. Add "Answer in one concise sentence." to the prompt.
3. **Low Context Recall** — Increase k, or check that your ground truth answers only reference what is actually in the indexed documents.

**Key lesson:** Writing good ground truth is harder than it looks. The reference must match the knowledge base exactly, not your general domain knowledge.

In [ ]:
# Exercise 3 — sample solution structure (space knowledge base)
sample_docs = [
    Document(
        page_content="The Moon orbits Earth about once every 27.3 days. It is Earth's only natural satellite.",
        metadata={"source": "space", "topic": "moon"},
    ),
    Document(
        page_content="Apollo 11 landed on the Moon on July 20, 1969. Neil Armstrong was the first human to walk on the Moon.",
        metadata={"source": "space", "topic": "apollo"},
    ),
    Document(
        page_content="Mars has two small moons named Phobos and Deimos. Mars has the largest volcano in the Solar System: Olympus Mons.",
        metadata={"source": "space", "topic": "mars"},
    ),
    Document(
        page_content="Jupiter is the largest planet in the Solar System, 2.5 times the mass of all other planets combined.",
        metadata={"source": "space", "topic": "jupiter"},
    ),
    Document(
        page_content="Saturn is known for its large ring system made of ice and rock. Saturn has 146 known moons.",
        metadata={"source": "space", "topic": "saturn"},
    ),
]

sample_test_set = [
    {
        "question": "How long does the Moon take to orbit Earth?",
        "ground_truth": "The Moon orbits Earth about once every 27.3 days.",
    },
    {
        "question": "Who first walked on the Moon?",
        "ground_truth": "Neil Armstrong was the first human to walk on the Moon, during Apollo 11 in 1969.",
    },
    {
        "question": "What are the moons of Mars called?",
        "ground_truth": "Mars has two moons named Phobos and Deimos.",
    },
    {
        "question": "What is Saturn known for?",
        "ground_truth": "Saturn is known for its large ring system made of ice and rock.",
    },
]

sample_vs = Chroma.from_documents(sample_docs, embedding=embeddings)
sample_ret = sample_vs.as_retriever(search_kwargs={"k": 2})
sample_chain = (
    {"context": sample_ret | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

sample_rows = []
for item in sample_test_set:
    q = item["question"]
    ctxs = sample_ret.invoke(q)
    resp = sample_chain.invoke(q)
    sample_rows.append(
        {
            "user_input": q,
            "reference": item["ground_truth"],
            "retrieved_contexts": [c.page_content for c in ctxs],
            "response": resp,
        }
    )

result_sample = evaluate(
    Dataset.from_list(sample_rows),
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)
print("Space knowledge base evaluation:")
df_sample = result_sample.to_pandas()
print(df_sample[["user_input"] + METRIC_COLS].to_string(index=False))
print("\nMean scores:")
for col in METRIC_COLS:
    print(f"  {col:<22} {df_sample[col].mean():.3f}")

---

## Academic References

### Primary Sources

**RAGAS:**  
Es, S., James, J., Anke, L.E., & Schockaert, S. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation.* arXiv:2309.15217. https://arxiv.org/abs/2309.15217

**RAG (original paper):**  
Lewis, P., Perez, E., Piktus, A., Petroni, F., Karpukhin, V., Goyal, N., ... & Kiela, D. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* Advances in Neural Information Processing Systems, 33. https://arxiv.org/abs/2005.11401

**LLM Evaluation Survey:**  
Chang, Y., Wang, X., Wang, J., Wu, Y., Yang, L., Zhu, K., ... & Xie, X. (2024). *A Survey on Evaluation of Large Language Models.* ACM Transactions on Intelligent Systems and Technology, 15(3). https://arxiv.org/abs/2307.03109

**RAG Survey:**  
Gao, Y., Xiong, Y., Gao, X., Jia, K., Pan, J., Bi, Y., ... & Wang, H. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* arXiv:2312.10997. https://arxiv.org/abs/2312.10997

### Further Reading

- RAGAS documentation: https://docs.ragas.io
- LangChain RAGAS integration: https://docs.ragas.io/en/stable/howtos/integrations/langchain.html
- RAGAS test set generation: https://docs.ragas.io/en/stable/concepts/test_dataset_generation.html
- BERTScore (Zhang et al. 2019): https://arxiv.org/abs/1904.09675 — alternative similarity metric
- TruLens (Snowflake): https://www.trulens.org — alternative RAG evaluation framework